# Chapman mechanism and the ozone layer

*chemistry · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/chemistry/chapman_ozone)

Four reactions between O, O₂ and O₃, driven by sunlight, make an ozone layer. Written as a reaction network $\dot c = S\,r(c)$ with a $3\times4$ stoichiometry matrix, the stiff system is integrated by backward Euler with Newton steps at every altitude at once, from sunrise to centuries later: O and O₃ trade places within seconds, odd oxygen takes days to centuries to settle, and the steady state peaks near 30 km.

**Vector form**

$$\frac{d\mathbf c}{dt} = S\,\mathbf r(\mathbf c),\qquad \mathbf r = \big(j_1[\mathrm{O_2}],\ k_2[\mathrm M][\mathrm O][\mathrm{O_2}],\ j_3[\mathrm{O_3}],\ k_4[\mathrm O][\mathrm{O_3}]\big),\qquad \mathbf a^{\mathsf T}S = 0,\qquad [\mathrm{O_3}] \approx [\mathrm{O_2}]\sqrt{\frac{j_1 k_2 [\mathrm M]}{j_3 k_4}}$$

**Index (tensor) form**

$$\dot c_{as} = S_{sr}\,r_{ar}(c_a) \qquad a_s S_{sr} = 0 \qquad c^{+}_{as} - c_{as} - \Delta t\,S_{sr}\,r_{ar}(c^{+}_a) = 0 \qquad J_{asc} = S_{sr}\,\frac{\partial r_{ar}}{\partial c_{ac}} \qquad \big(\delta_{sc} - \Delta t\,J_{asc}\big)\,\delta c_{ac} = -G_{as}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $c_{as}$ | `c` | (A, 3) | number densities of $\mathrm O$, $\mathrm{O_2}$, $\mathrm{O_3}$ at altitude $a$, in cm$^{-3}$ |
| $S_{sr}$ | `S` | (3, 4) | net molecules of species $s$ made by reaction $r$; column 1 is $(2, -1, 0)$ for $\mathrm{O_2} \to 2\,\mathrm O$ |
| $a_s S_{sr}$ | `ATOMS @ S` | (4,) | repeated $s$ ⇒ summed: oxygen atoms made by each reaction, all zero |
| $r_{ar}$ | `rates(c, j1, j3, k2, k4, M)` | (A, 4) | mass-action rates of the four reactions |
| $S_{sr}r_{ar}$ | `einsum('sr,ar->as', S, r)` | (A, 3) | repeated $r$ ⇒ summed: $dc/dt$ |
| $J_{asc}$ | `einsum('sr,arc->asc', S, dr_dc)` | (A, 3, 3) | Jacobian of the network, repeated $r$ summed |
| $\delta c_{ac}$ | `torch.linalg.solve(eye - dt * J, -G)` | (A, 3) | one Newton update of the backward-Euler step, a $3\times3$ solve per altitude |
| $k_2(T),\ k_4(T)$ | `k2_rate(T), k4_rate(T)` | (A,) | $6.0\times10^{-34}(T/300)^{-2.4}$ cm$^6$ s$^{-1}$ and $8.0\times10^{-12}e^{-2060/T}$ cm$^3$ s$^{-1}$ (JPL, approximate) |
| $j_1(z),\ j_3(z)$ | `photolysis_O2(z, mu), photolysis_O3(O3, z, mu)` | (A,) | photolysis rates attenuated by the $\mathrm{O_2}$ and $\mathrm{O_3}$ columns overhead (illustrative one-band model) |
| $[\mathrm{O_3}]_{\text{ss}}$ | `steady_state(j1, j3, k2, k4, M, O2)` | (A,) | exact Chapman steady state from a quadratic in $[\mathrm O]$ |

Sydney Chapman (1930) explained the ozone layer with four reactions. Ultraviolet light below 242 nm splits $\mathrm{O_2}$ (1); each O atom quickly sticks to an $\mathrm{O_2}$ with a third body $\mathrm M$ to carry the energy away (2); ozone absorbs ultraviolet and visible light and falls apart again (3), which is how it shields the ground, but this does not destroy "odd oxygen" $\mathrm O_x = \mathrm O + \mathrm{O_3}$; only $\mathrm O + \mathrm{O_3} \to 2\,\mathrm{O_2}$ (4) does. Reactions (2) and (3) cycle O and $\mathrm{O_3}$ in milliseconds to minutes, while (1) and (4) set the total over days in the upper stratosphere and decades or more low down: a stiff system with time scales $10^{14}$ apart, so an explicit step would need billions of steps, while backward Euler (stable for any $\Delta t$) takes 300 steps spaced evenly in $\log t$. Every column of $S$ conserves oxygen atoms, and backward Euler keeps such linear invariants exactly. Balancing the two pairs gives $[\mathrm{O_3}] \approx [\mathrm{O_2}]\sqrt{j_1 k_2[\mathrm M]/(j_3 k_4)}$: going down, the air (and with it $[\mathrm{O_2}]^{3/2}$) gets denser exponentially while the $\mathrm{O_2}$-splitting light is used up, and the product peaks where the slant $\mathrm{O_2}$ optical depth is about 3. The rate constants $k_2$, $k_4$ are the JPL evaluation values (approximate); the photolysis rates use a single effective band per molecule with illustrative parameters, a stand-in for an integration over the solar spectrum, and the sun shines all the time. Below about 18 km the odd-oxygen lifetime exceeds even the longest run, so those levels never settle; in the real atmosphere winds carry ozone there. Chapman chemistry alone gives too much ozone (here a peak near $8\times10^{12}$ cm$^{-3}$ and about 380 DU above 15 km, against roughly $5\times10^{12}$ cm$^{-3}$ near 20–25 km and 300 DU observed). The missing sinks are catalytic cycles $\mathrm X + \mathrm{O_3} \to \mathrm{XO} + \mathrm{O_2}$, $\mathrm{XO} + \mathrm O \to \mathrm X + \mathrm{O_2}$, which do reaction (4) many times over per catalyst: NOₓ (Crutzen, 1970), and chlorine from CFCs (Molina and Rowland, 1974), the chemistry behind the ozone hole and the Montreal Protocol (Nobel Prize 1995).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/chemistry/chapman_ozone/main.py
"""Main loop: altitudes → rate coefficients → backward-Euler network S·r(c) from sunrise → steady ozone layer → GIF."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import ATOMS, SPECIES, S, chapman_approx, steady_state


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    n_levels = 10 if fast else params["levels"]
    mu = math.cos(math.radians(params["sza"]))
    z = tensors.altitudes(n_levels, dev)  # (A,)
    times = tensors.time_grid(10.0 ** params["log10_t_end"], 30 if fast else 300)  # (T,)
    print(f"Chapman network at {n_levels} altitudes, 15–60 km, sun at {params['sza']:.0f}° from the zenith")
    print("S (rows " + ", ".join(SPECIES) + "; columns reactions 1–4):")
    print(S.numpy())
    print(f"oxygen atoms per species a = {ATOMS.tolist()},  a·S = {(ATOMS @ S).tolist()}  (every reaction conserves O)")

    threads = torch.get_num_threads()
    torch.set_num_threads(1)  # 3 × 3 solves per level: one thread is far steadier than many on a busy CPU
    try:
        c, (j1, j3, k2, k4, M) = solve.solve(z, times, mu)  # (T, A, 3)
    finally:
        torch.set_num_threads(threads)
    atoms = c @ ATOMS.to(c)  # (T, A) oxygen atoms per cm³
    print(
        f"{len(times) - 1} backward-Euler steps to t = {times[-1]:.0e} s; max relative change of O atoms "
        f"{((atoms - atoms[0]) / atoms[0]).abs().max():.1e}"
    )

    O2 = c[-1, :, 1]
    O_ss, O3_ss = steady_state(j1, j3, k2, k4, M, O2)
    approx = chapman_approx(j1, j3, k2, k4, M, O2)
    fast_tau, slow_tau = solve.lifetimes(c[-1], j1, j3, k2, k4, M)
    print(" z/km   j1/s⁻¹   j3/s⁻¹   [O3] final   exact SS   √-formula   [O]/[O3]   τ_fast/s   τ_Ox/s")
    for a in range(0, n_levels, max(1, n_levels // 9)):
        print(
            f"{z[a]:5.1f}  {j1[a]:8.1e} {j3[a]:8.1e}  {c[-1, a, 2]:10.3e} {O3_ss[a]:10.3e} {approx[a]:10.3e}"
            f"   {c[-1, a, 0] / c[-1, a, 2]:8.1e}  {fast_tau[a]:8.1e}  {slow_tau[a]:8.1e}"
        )
    done = slow_tau < times[-1] / 30  # levels that have had many odd-oxygen lifetimes to settle
    err = ((c[-1, done, 2] - O3_ss[done]) / O3_ss[done]).abs().max()
    print(f"levels with τ_Ox < t_end / 30: final [O3] matches the exact steady state to {err:.1e}")
    peak = int(c[-1, :, 2].argmax())
    column = (c[-1, :, 2].sum() * (z[1] - z[0]) * 1e5 / 2.687e16).item()
    print(
        f"ozone peak {c[-1, peak, 2]:.2e} cm⁻³ at {z[peak]:.1f} km; column above 15 km {column:.0f} DU "
        f"(observed: peak ≈ 5e12 cm⁻³ near 20–25 km, total ≈ 300 DU)"
    )
    return plot.animate(times, z, c, O3_ss, fast_tau, slow_tau, out_dir / "chapman_ozone.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/chemistry/chapman_ozone/equation.py
r"""Physics → tensor translation: the Chapman mechanism as a chemical reaction network dc/dt = S·r(c).

Indices: a altitude level, s species (O, O₂, O₃), r reaction (1–4), e element (oxygen).

    (1) O₂ + hν → 2 O             r₁ = j₁ [O₂]
    (2) O + O₂ + M → O₃ + M       r₂ = k₂ [M] [O] [O₂]
    (3) O₃ + hν → O + O₂          r₃ = j₃ [O₃]
    (4) O + O₃ → 2 O₂             r₄ = k₄ [O] [O₃]

    dc_as/dt = S_sr r_ar(c_a)                         (r summed)
The stoichiometry matrix S (species × reactions) holds the net number of molecules of s made by reaction r.
Atoms are conserved because the composition row a_e = (1, 2, 3) oxygen atoms per (O, O₂, O₃) is a left null
vector of S:  a_s S_sr = 0 for every r (s summed).

Backward Euler, c⁺ = c + Δt S r(c⁺), is solved by Newton's method with the Jacobian
    J_asc = S_sr ∂r_ar/∂c_ac                          (r summed),
a 3 × 3 linear solve per altitude. It is stable for any step, which matters because the fast O ⇄ O₃ cycling
(milliseconds to hours) and the slow build-up of odd oxygen Oₓ = O + O₃ (days to centuries) differ by up to 10¹⁴.
"""

import torch

SPECIES = ["O", "O2", "O3"]
S = torch.tensor(
    [
        # r1  r2  r3  r4
        [2.0, -1.0, 1.0, -1.0],  # O
        [-1.0, -1.0, 1.0, 2.0],  # O2
        [0.0, 1.0, -1.0, -1.0],  # O3
    ],
    dtype=torch.float64,
)
ATOMS = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)  # oxygen atoms in O, O2, O3


def rates(c: torch.Tensor, j1, j3, k2, k4, M) -> torch.Tensor:
    """r_ar of the four reactions at every altitude.   c (A, 3), coefficients (A,) → (A, 4), cm⁻³ s⁻¹"""
    o, o2, o3 = c[:, 0], c[:, 1], c[:, 2]  # [O], [O₂], [O₃]
    return torch.stack([j1 * o2, k2 * M * o * o2, j3 * o3, k4 * o * o3], -1)


def rate_jacobian(c: torch.Tensor, j1, j3, k2, k4, M) -> torch.Tensor:
    """∂r_ar/∂c_ac.   (A, 4, 3)"""
    o, o2, o3 = c[:, 0], c[:, 1], c[:, 2]  # [O], [O₂], [O₃]
    z = torch.zeros_like(o)
    return torch.stack(
        [
            torch.stack([z, j1, z], -1),  # r1 = j1 [O2]
            torch.stack([k2 * M * o2, k2 * M * o, z], -1),  # r2 = k2 [M][O][O2]
            torch.stack([z, z, j3], -1),  # r3 = j3 [O3]
            torch.stack([k4 * o3, z, k4 * o], -1),  # r4 = k4 [O][O3]
        ],
        -2,
    )


def dcdt(c: torch.Tensor, *coeffs) -> torch.Tensor:
    """dc_as/dt = S_sr r_ar, r summed.   (A, 3)"""
    return torch.einsum("sr,ar->as", S.to(c), rates(c, *coeffs))


def jacobian(c: torch.Tensor, *coeffs) -> torch.Tensor:
    """J_asc = S_sr ∂r_ar/∂c_ac, r summed.   (A, 3, 3)"""
    return torch.einsum("sr,arc->asc", S.to(c), rate_jacobian(c, *coeffs))


def k2_rate(T: torch.Tensor) -> torch.Tensor:
    """O + O₂ + M → O₃ + M, low-pressure limit 6.0×10⁻³⁴ (T/300)^-2.4 cm⁶ s⁻¹ (JPL evaluation, approximate)."""
    return 6.0e-34 * (T / 300) ** -2.4


def k4_rate(T: torch.Tensor) -> torch.Tensor:
    """O + O₃ → 2 O₂, 8.0×10⁻¹² exp(-2060 / T) cm³ s⁻¹ (JPL evaluation, approximate)."""
    return 8.0e-12 * torch.exp(-2060 / T)


def steady_state(j1, j3, k2, k4, M, O2) -> tuple[torch.Tensor, torch.Tensor]:
    """Exact Chapman steady state with [O₂] fixed.

    d[Oₓ]/dt = 0 gives k₄[O][O₃] = j₁[O₂]; d[O₃]/dt = 0 gives k₂[M][O][O₂] = [O₃](j₃ + k₄[O]).
    Eliminating [O₃]:  k₂k₄[M] [O]² - j₁k₄ [O] - j₁j₃ = 0, so
        [O] = (j₁k₄ + √(j₁²k₄² + 4 k₂k₄[M] j₁j₃)) / (2 k₂k₄[M]),   [O₃] = j₁[O₂] / (k₄[O])."""
    o = (j1 * k4 + torch.sqrt((j1 * k4) ** 2 + 4 * k2 * k4 * M * j1 * j3)) / (2 * k2 * k4 * M)
    return o, j1 * O2 / (k4 * o)


def chapman_approx(j1, j3, k2, k4, M, O2) -> torch.Tensor:
    """The textbook form [O₃] = [O₂] √(j₁ k₂ [M] / (j₃ k₄)), valid when k₄[O] ≪ j₃."""
    return O2 * torch.sqrt(j1 * k2 * M / (j3 * k4))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/chemistry/chapman_ozone/tensors.py
"""Build the concrete tensors: the altitude grid, an exponential atmosphere, temperatures and photolysis rates.

Units: cm⁻³ for number densities, s for time, km for altitude.

Photolysis is modelled with one effective absorption band per molecule, attenuated by the column overhead along
the slanted sun path (Beer–Lambert). These band values are illustrative choices that give photolysis rates of
roughly the right size (j_O₂ ~ 10⁻¹⁰ s⁻¹ at 50 km falling to ~ 10⁻¹² s⁻¹ near 25 km; j_O₃ ~ 10⁻² s⁻¹ at the top
falling to ~ 4×10⁻⁴ s⁻¹ below the ozone layer); a real calculation integrates over the solar spectrum.
"""

import math

import torch

N0 = 2.55e19  # air number density at sea level, cm⁻³
H_SCALE = 7.0  # scale height, km
X_O2 = 0.21  # O₂ mole fraction

J1_TOP, SIGMA_O2 = 2e-10, 5e-23  # O₂ band: unattenuated rate s⁻¹, effective cross-section cm² (illustrative)
J3_UV, SIGMA_O3 = 8e-3, 1e-17  # O₃ Hartley band (cross-section ≈ 1.1×10⁻¹⁷ cm² at the 255 nm peak)
J3_VIS = 4e-4  # O₃ Chappuis band (visible light, practically unattenuated), s⁻¹


def altitudes(n: int, device: torch.device) -> torch.Tensor:
    """Levels from 15 to 60 km.   (A,)"""
    return torch.linspace(15.0, 60.0, n, dtype=torch.float64, device=device)


def air_density(z: torch.Tensor) -> torch.Tensor:
    """Exponential atmosphere n(z) = n₀ exp(-z / H), cm⁻³.   (A,)"""
    return N0 * torch.exp(-z / H_SCALE)


def temperature(z: torch.Tensor) -> torch.Tensor:
    """US Standard Atmosphere 1976 temperature profile (piecewise linear in z), K.   (A,)"""
    z_knots = torch.tensor([0.0, 11.0, 20.0, 32.0, 47.0, 51.0, 71.0, 85.0], dtype=z.dtype, device=z.device)
    T_knots = torch.tensor(
        [288.15, 216.65, 216.65, 228.65, 270.65, 270.65, 214.65, 186.95], dtype=z.dtype, device=z.device
    )
    i = (torch.searchsorted(z_knots, z) - 1).clamp(0, len(z_knots) - 2)
    w = (z - z_knots[i]) / (z_knots[i + 1] - z_knots[i])
    return T_knots[i] + w * (T_knots[i + 1] - T_knots[i])


def column_above(n: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    """Number of molecules per cm² above each level: a sum over the levels above plus half the own layer,
    plus the part above the top level, n_top H (exponential tail).   n, z (A,) → (A,)"""
    dz = (z[1] - z[0]) * 1e5  # km → cm
    above = n.flip(0).cumsum(0).flip(0) - 0.5 * n  # levels at and above, own level half
    return above * dz + n[-1] * H_SCALE * 1e5


def photolysis_O2(z: torch.Tensor, mu: float) -> torch.Tensor:
    """j₁ = J1_TOP exp(-σ_O₂ N_O₂ / μ): O₂ shields itself.   (A,)"""
    N_O2 = X_O2 * air_density(z) * H_SCALE * 1e5  # exact O₂ column above z for an exponential atmosphere
    return J1_TOP * torch.exp(-SIGMA_O2 * N_O2 / mu)


def photolysis_O3(O3: torch.Tensor, z: torch.Tensor, mu: float) -> torch.Tensor:
    """j₃ = J3_VIS + J3_UV exp(-σ_O₃ N_O₃ / μ): the ozone overhead shields the UV part.   (A,)"""
    return J3_VIS + J3_UV * torch.exp(-SIGMA_O3 * column_above(O3, z) / mu)


def initial_state(z: torch.Tensor) -> torch.Tensor:
    """Sunrise on an atmosphere with all its oxygen as O₂: c = ([O], [O₂], [O₃]) = (0, 0.21 n, 0).   (A, 3)"""
    c = torch.zeros(len(z), 3, dtype=torch.float64, device=z.device)
    c[:, 1] = X_O2 * air_density(z)
    return c


def time_grid(t_end: float, n: int) -> torch.Tensor:
    """Logarithmically spaced times from 10⁻⁴ s to t_end, with t = 0 first.   (n + 1,)"""
    return torch.cat(
        [torch.zeros(1, dtype=torch.float64), torch.logspace(-4, math.log10(t_end), n, dtype=torch.float64)]
    )


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/chemistry/chapman_ozone/solve.py
"""Backward-Euler integration of the stiff Chapman network at every altitude at once, with Newton iterations."""

import torch

from problems._common.progress import steps

from . import tensors
from .equation import dcdt, jacobian, k2_rate, k4_rate


def backward_euler_step(c, dt: float, coeffs, n_newton: int = 6):
    """Solve G(c⁺) = c⁺ - c - Δt S r(c⁺) = 0 by Newton: (1 - Δt J) δ = -G, c⁺ ← c⁺ + δ, all altitudes batched."""
    eye = torch.eye(3, dtype=c.dtype, device=c.device)
    new = c.clone()
    for _ in range(n_newton):
        G = new - c - dt * dcdt(new, *coeffs)  # (A, 3)
        delta = torch.linalg.solve(eye - dt * jacobian(new, *coeffs), -G)  # (A, 3)
        new = (new + delta).clamp(min=0.0)
    return new


def solve(z: torch.Tensor, times: torch.Tensor, mu: float):
    """Integrate from sunrise (all O₂) through the log-spaced times (T,).

    j₁, k₂, k₄ and [M] are fixed per altitude; j₃ depends on the ozone column overhead and is updated from the
    previous step (lagged), which couples the levels only through the shielding.
    Returns c (T, A, 3) and the final coefficients (j1, j3, k2, k4, M), each (A,)."""
    M = tensors.air_density(z)
    T = tensors.temperature(z)
    j1, k2, k4 = tensors.photolysis_O2(z, mu), k2_rate(T), k4_rate(T)
    c = tensors.initial_state(z)
    out = [c]
    for k in steps(len(times) - 1, "backward-Euler steps"):
        j3 = tensors.photolysis_O3(c[:, 2], z, mu)
        c = backward_euler_step(c, (times[k + 1] - times[k]).item(), (j1, j3, k2, k4, M))
        out.append(c)
    j3 = tensors.photolysis_O3(c[:, 2], z, mu)
    return torch.stack(out), (j1, j3, k2, k4, M)


def lifetimes(c: torch.Tensor, j1, j3, k2, k4, M):
    """Time scales at the end state, each (A,), in s:
    fast  1 / (k₂[M][O₂] + j₃): how quickly O and O₃ trade places;
    slow  [Oₓ] / (2 k₄[O][O₃]): how long odd oxygen lives before O + O₃ destroys it."""
    o, o2, o3 = c[:, 0], c[:, 1], c[:, 2]  # [O], [O₂], [O₃]
    fast = 1 / (k2 * M * o2 + j3)
    slow = (o + o3) / (2 * k4 * o * o3)
    return fast, slow


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/chemistry/chapman_ozone/plot.py
"""Animate the ozone layer forming after sunrise: profiles, time series at three heights and the two time scales."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif

BLUE, RED, GREY, AMBER, DARK, GREEN = "#2563eb", "#dc2626", "#9ca3af", "#f59e0b", "#374151", "#16a34a"


def _when(t: float) -> str:
    for unit, s in (("yr", 3.156e7), ("days", 86400), ("h", 3600), ("min", 60), ("s", 1)):
        if t >= s:
            return f"{t / s:.3g} {unit}"
    return f"{t:.1e} s"


def _decades(ax, axis: str, lo: int, hi: int, step: int = 2):
    ticks = [10.0**k for k in range(lo, hi + 1, step)]
    labels = [f"1e{k}" for k in range(lo, hi + 1, step)]  # plain labels: no mathtext to typeset per frame
    if axis == "x":
        ax.set_xscale("log")
        ax.set_xticks(ticks, labels)
        ax.set_xlim(ticks[0], ticks[-1])
    else:
        ax.set_yscale("log")
        ax.set_yticks(ticks, labels)
        ax.set_ylim(ticks[0], ticks[-1])
    ax.minorticks_off()


def animate(times, z, c, O3_ss, fast_tau, slow_tau, path: Path) -> Path:
    """times (T,) s; z (A,) km; c (T, A, 3) of (O, O2, O3) in cm⁻³; steady state and time scales (A,)."""
    t = times.numpy()
    zs = z.cpu().numpy()
    C = c.cpu().numpy()
    ss, tf, tx = O3_ss.cpu().numpy(), fast_tau.cpu().numpy(), slow_tau.cpu().numpy()
    tt = np.maximum(t, 1e-5)  # t = 0 drawn at the left edge of the log axis
    floor = 1e2

    fig, (ax_z, ax_t, ax_l) = plt.subplots(1, 3, figsize=(10, 4.4), gridspec_kw={"width_ratios": [1, 1.25, 1]})
    fig.subplots_adjust(left=0.065, right=0.98, top=0.88, bottom=0.13, wspace=0.32)

    # --- altitude profiles ---
    ax_z.plot(ss, zs, color=DARK, lw=1.2, ls=(0, (4, 3)), label="O₃ exact steady state")
    (l_o3,) = ax_z.plot([], [], color=BLUE, lw=2, label="O₃")
    (l_o,) = ax_z.plot([], [], color=RED, lw=1.5, label="O")
    _decades(ax_z, "x", 2, 14)
    ax_z.set_ylim(zs[0], zs[-1])
    ax_z.set_xlabel("number density (cm⁻³)", labelpad=1)
    ax_z.set_ylabel("altitude (km)", labelpad=1)
    ax_z.legend(fontsize=7, loc="upper left", frameon=False)
    title = ax_z.set_title("", fontsize=9)

    # --- time series at three heights ---
    picks = [int(np.abs(zs - h).argmin()) for h in (20, 30, 45)]
    shades = ["#93c5fd", BLUE, "#1e3a8a"]
    for a, col in zip(picks, shades):
        ax_t.plot(tt, np.maximum(C[:, a, 2], floor), color=col, lw=1.8, label=f"O₃ {zs[a]:.0f} km")
        ax_t.plot(tt, np.maximum(C[:, a, 0], floor), color=col, lw=1, ls=(0, (3, 2)))
        ax_t.plot([tf[a]], [floor * 3], "v", color=AMBER, ms=6, mec=DARK, mew=0.5)
        if tx[a] < t[-1]:
            ax_t.plot([tx[a]], [floor * 3], "v", color=GREEN, ms=6, mec=DARK, mew=0.5)
    _decades(ax_t, "x", -4, int(np.ceil(np.log10(t[-1]))))
    _decades(ax_t, "y", 2, 14)
    ax_t.set_xlabel("time since sunrise (s)", labelpad=1)
    ax_t.set_title("solid O₃, dashed O; ▼ amber τ_fast, ▼ green τ_Oₓ", fontsize=9)
    ax_t.legend(fontsize=7, loc="upper left", frameon=False)
    (cursor,) = ax_t.plot([1e-4, 1e-4], [1e2, 1e14], color=GREY, lw=1)

    # --- time scales: fast O ⇄ O₃ exchange, slow odd-oxygen relaxation ---
    ax_l.plot(tf, zs, color=AMBER, lw=2, label="τ_fast: O ⇄ O₃")
    ax_l.plot(tx, zs, color=GREEN, lw=2, label="τ_Oₓ: odd oxygen")
    for label, sec, y in (("1 min", 60, 1), ("1 day", 86400, 1), ("1 yr", 3.156e7, 4)):
        ax_l.axvline(sec, color="#e5e7eb", lw=1)
        ax_l.text(sec, zs[-1] - y, label, fontsize=7, color="#6b7280", ha="center", va="top")
    _decades(ax_l, "x", -4, 12, step=4)
    ax_l.set_ylim(zs[0], zs[-1])
    ax_l.set_xlabel("time scale (s)", labelpad=1)
    ax_l.legend(fontsize=7, loc="lower left", frameon=False)
    ax_l.set_title("left of the cursor: settled", fontsize=9)
    (cursor_l,) = ax_l.plot([1e-4, 1e-4], [zs[0], zs[-1]], color=GREY, lw=1)
    fig.suptitle("Chapman ozone layer: dc/dt = S·r(c), 4 reactions × 3 species at every altitude", fontsize=10)

    def update(k: int):
        l_o3.set_data(np.maximum(C[k, :, 2], floor), zs)
        l_o.set_data(np.maximum(C[k, :, 0], floor), zs)
        title.set_text(f"profiles, t = {_when(t[k])}" if k else "profiles at sunrise (all O₂)")
        cursor.set_data([tt[k], tt[k]], [1e2, 1e14])
        cursor_l.set_data([tt[k], tt[k]], [zs[0], zs[-1]])

    return save_gif(fig, update, frame_indices(len(t), 60), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'sza': 30.0,
    'log10_t_end': 10,
    'levels': 46
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.chemistry.chapman_ozone")]:
    del sys.modules[name]
main = importlib.import_module("problems.chemistry.chapman_ozone.main")
problem_dir = Path("problems/chemistry/chapman_ozone")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
